# KG Pipeline — Message-Atomic (8 Stages)

This notebook walks the full knowledge-graph ingestion pipeline, one stage at a time.
Each cell is safe to re-run; most write operations are idempotent.

---

## Prerequisites

| Requirement | Description |
|---|---|
| Python ≥ 3.11 | Tested on 3.11/3.12 |
| `.env` file | At the project root (see below) |
| Postgres | Running and schema-ready |
| Redis | Running |
| Pinecone | API key + three indexes |
| OpenAI | API key with chat + embedding access |
| Telegram | `api_id`, `api_hash`, authenticated session |

---

## Required environment variables (`.env`)

```
TELEGRAM_API_ID=...
TELEGRAM_API_HASH=...
TELEGRAM_PHONE=...
DATABASE_URL=postgresql://...
REDIS_URL=redis://localhost:6379/0
PINECONE_API_KEY=...
PINECONE_INDEX_STORY=story-embeddings
PINECONE_INDEX_THEME=theme-centroids
PINECONE_INDEX_EVENT=event-centroids
OPENAI_API_KEY=...
```

---

## Pipeline stages and write targets

| # | Stage | Writes to |
|---|---|---|
| 1 | Setup & Connections | — |
| 2 | Telegram Client & Channel Discovery | Postgres (`channel_profiles`) |
| 3 | Message Ingestion | Postgres (`raw_messages`), Redis stream |
| 4 | Translation | Postgres (`raw_messages.english_text`), OpenAI API |
| 5 | Embedding | Pinecone (story index), Postgres (`message_embeddings`), OpenAI API |
| 6 | Extraction + Node Resolution | Postgres (`message_semantics`, `message_nodes`, `nodes`, `message_matches`), Pinecone (theme/event centroids), OpenAI API |
| 7 | Projection + Heat | Postgres (`node_relations`, `theme_daily_stats`, `node_heat_view`) |
| 8 | Query + Grouped View | — (read-only) |

> **Warning — live writes:** Sections 2–7 write to your Postgres database, Redis stream,
> and Pinecone indexes. Set `MESSAGE_LIMIT` conservatively (e.g. 50) for an initial test run.

---
## Section 1 — Environment Setup & Connections

In [ ]:
import sys
import os
from pathlib import Path

# Support running from notebooks/ subdirectory OR project root.
_nb_dir = Path().resolve()
_src_candidates = [_nb_dir / "src", _nb_dir.parent / "src"]
for _candidate in _src_candidates:
    if _candidate.exists() and str(_candidate) not in sys.path:
        sys.path.insert(0, str(_candidate))
        print(f"Added to sys.path: {_candidate}")
        break

import nest_asyncio
nest_asyncio.apply()

import pandas as pd
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 180)

print("Python:", sys.version)

In [ ]:
import time

from telegram_scraper.config import Settings
from telegram_scraper.kg.config import KGSettings

# Resolve .env — look in cwd and one directory up.
_env_candidates = [Path(".env"), Path("../.env")]
_env_path = next((p for p in _env_candidates if p.exists()), Path(".env"))

kg_settings = KGSettings.load(_env_path)
settings = Settings.load(_env_path)

# Fail fast: validate all required credentials are set.
kg_settings.require_database()
kg_settings.require_stream()
kg_settings.require_vector_store()
kg_settings.require_embeddings()
kg_settings.require_semantic_extraction()
kg_settings.require_translation()

print("KGSettings loaded OK")
print(f"  embedding_model      : {kg_settings.embedding_model}")
print(f"  semantic_model       : {kg_settings.semantic_model}")
print(f"  translation_model    : {kg_settings.translation_model}")
print(f"  pinecone_index_story : {kg_settings.pinecone_index_story}")
print(f"  pinecone_index_theme : {kg_settings.pinecone_index_theme}")
print(f"  pinecone_index_event : {kg_settings.pinecone_index_event}")

In [ ]:
from telegram_scraper.kg import runtime

_t0 = time.monotonic()

repository   = runtime.build_repository(kg_settings)
stream       = runtime.build_stream(kg_settings)
embedder     = runtime.build_embedder(kg_settings)
extractor    = runtime.build_semantic_extractor(kg_settings)
translator   = runtime.build_message_translator(kg_settings)
vector_store = runtime.build_vector_store(kg_settings)

repository.ensure_schema()
stream.ensure_group()

_elapsed_setup = time.monotonic() - _t0
print(f"Infrastructure ready in {_elapsed_setup:.2f}s")

# Smoke test — list channels already in the database.
existing_channels = repository.list_channels()
print(f"Existing channels in DB: {len(existing_channels)}")
for ch in existing_channels:
    print(f"  [{ch.channel_id}] {ch.channel_title or '(no title)'} — {ch.message_count} messages")

---
## Section 2 — Telegram Client & Channel Discovery

In [ ]:
from telegram_scraper.telegram_client import TelegramAccountClient
from telegram_scraper.chat_discovery import discover_chats, filter_chats
from telegram_scraper.models import ChatType
from telegram_scraper.kg.services import KGProfileService

_t2 = time.monotonic()

telegram_client = TelegramAccountClient(
    api_id=settings.api_id,
    api_hash=settings.api_hash,
    phone=settings.phone,
    session_path=str(settings.session_path),
)
await telegram_client.connect()
print("Telegram client connected.")

dialogs = await telegram_client.get_dialogs()
all_chats = discover_chats(dialogs)
channels = filter_chats(all_chats, chat_types=(ChatType.CHANNEL,))
print(f"Found {len(channels)} channels.")

In [ ]:
# Display channels as a DataFrame.
channel_rows = [
    {
        "#": i,
        "chat_id": ch.chat_id,
        "title": ch.title or "",
        "username": ch.username or "",
        "slug": ch.slug or "",
    }
    for i, ch in enumerate(channels)
]
pd.DataFrame(channel_rows)

In [ ]:
# ── Configure these two variables ─────────────────────────────────────────────
CHANNEL_INDEX = 0    # row number from the table above
MESSAGE_LIMIT = 50   # how many recent messages to fetch
# ────────────────────────────────────────────────────────────────────────

selected_chat = channels[CHANNEL_INDEX]
print(f"Selected channel: [{selected_chat.chat_id}] {selected_chat.title} (@{selected_chat.username})")

# Sync channel profile to Postgres.
profile_service = KGProfileService(repository)
profile_service.sync_chat_metadata([selected_chat])
channel_profile = repository.get_channel_profile(selected_chat.chat_id)
print("Channel profile stored:")
print(f"  channel_title    : {channel_profile.channel_title if channel_profile else '---'}")
print(f"  channel_slug     : {channel_profile.channel_slug if channel_profile else '---'}")
print(f"  channel_username : {channel_profile.channel_username if channel_profile else '---'}")

_elapsed_channel = time.monotonic() - _t2
print(f"Section 2 done in {_elapsed_channel:.2f}s")

---
## Section 3 — Message Ingestion

In [ ]:
from telegram_scraper.kg.normalization import normalize_message_record
from telegram_scraper.kg.models import RawMessage

_t3 = time.monotonic()

raw_envelopes = []
async for envelope in telegram_client.iter_message_envelopes(
    selected_chat, limit=MESSAGE_LIMIT, reverse=False
):
    raw_envelopes.append(envelope)

print(f"Fetched {len(raw_envelopes)} envelopes from Telegram (newest first).")

# Normalize and sort chronologically.
raw_messages: list[RawMessage] = [
    normalize_message_record(env.record, raw_json=env.raw_json)
    for env in raw_envelopes
]
raw_messages.sort(key=lambda m: m.timestamp)
print(f"Normalized and sorted {len(raw_messages)} messages (oldest first).")

In [ ]:
# Persist to Postgres.
repository.upsert_raw_messages(raw_messages)
print(f"Upserted {len(raw_messages)} messages to Postgres.")

# Push to Redis stream.
for msg in raw_messages:
    stream.add(msg)
print(f"Pushed {len(raw_messages)} messages to Redis stream '{kg_settings.stream_key}'.")

_elapsed_ingest = time.monotonic() - _t3
print(f"Section 3 done in {_elapsed_ingest:.2f}s")

# Preview.
preview_rows = [
    {
        "message_id": m.message_id,
        "timestamp": m.timestamp.strftime("%Y-%m-%d %H:%M"),
        "sender": m.sender_name or str(m.sender_id or ""),
        "text_preview": (m.text or "")[:80] + ("..." if len(m.text or "") > 80 else ""),
        "has_media": bool(m.media_refs),
        "forwarded": m.forwarded_from is not None,
    }
    for m in raw_messages[:20]
]
pd.DataFrame(preview_rows)

---
## Section 4 — Translation

In [ ]:
_t4 = time.monotonic()

translated_messages = translator.translate_messages(raw_messages)
repository.save_raw_message_translations(translated_messages)

_elapsed_translate = time.monotonic() - _t4
print(f"Translated {len(translated_messages)} messages in {_elapsed_translate:.2f}s")

# Language distribution.
lang_series = pd.Series(
    [m.source_language or "en" for m in translated_messages],
    name="source_language",
)
print("\nLanguage distribution:")
print(lang_series.value_counts().to_string())

In [ ]:
# Before / after samples for non-English messages.
non_english = [
    m for m in translated_messages
    if m.source_language and m.source_language not in ("en", "english") and m.english_text
]
print(f"Non-English messages with translations: {len(non_english)}")

sample_rows = [
    {
        "message_id": m.message_id,
        "language": m.source_language,
        "original": (m.text or "")[:120],
        "english": (m.english_text or "")[:120],
    }
    for m in non_english[:10]
]
pd.DataFrame(sample_rows)

---
## Section 5 — Embedding

Each message is embedded independently using the configured embedding model.
Vectors are upserted to Pinecone (story index) and the `message_embeddings` table tracks
which messages have been embedded and with which model version.

In [ ]:
from datetime import datetime, timezone
from telegram_scraper.kg.models import MessageEmbeddingRecord
from telegram_scraper.kg.extraction import preferred_message_text, safe_message_text

_t5 = time.monotonic()

# Filter to messages that have text content.
embeddable = [m for m in translated_messages if (m.english_text or m.text or "").strip()]
print(f"Messages with text to embed: {len(embeddable)} / {len(translated_messages)}")

# Build embedding texts.
embed_texts = [
    safe_message_text(
        preferred_message_text(m) or "(media only)",
        max_chars=kg_settings.semantic_max_chars,
    )
    for m in embeddable
]

# Compute embeddings in one batch.
vectors = embedder.embed_texts(embed_texts)
print(f"Embedding dimensionality: {len(vectors[0]) if vectors else 0}")

# Build records and upsert to Pinecone.
embedding_records = [
    MessageEmbeddingRecord(
        channel_id=m.channel_id,
        message_id=m.message_id,
        embedding=vec,
        timestamp=m.timestamp,
        node_ids=(),
    )
    for m, vec in zip(embeddable, vectors)
]

vector_store.upsert_message_embeddings(embedding_records)
print(f"Upserted {len(embedding_records)} message embeddings to Pinecone.")

# Mark embedded in Postgres.
for rec in embedding_records:
    repository.mark_message_embedded(
        channel_id=rec.channel_id,
        message_id=rec.message_id,
        version=kg_settings.embedding_model,
    )

_elapsed_embed = time.monotonic() - _t5
print(f"Section 5 done in {_elapsed_embed:.2f}s ({len(embedding_records)/max(_elapsed_embed, 0.001):.1f} msg/s)")

---
## Section 6 — Extraction + Node Resolution

`KGNodeProcessingService.process_messages()` runs the full extraction-and-resolution loop:

1. Calls `extractor.extract_message(msg)` per message (OpenAI structured output)
2. Iterates extracted candidates (events, people, nations, orgs, places, themes)
3. Uses `NodeResolver.resolve_message()` to match or create nodes in Postgres
4. Writes `MessageNodeAssignment` records linking each message to its nodes
5. Updates node centroid vectors in Pinecone (theme + event indexes)
6. Does cross-channel matching via `vector_store.query_message_embeddings()`
7. Calls `hierarchy_service.rebuild()` at the end to update event parent/child structure

Messages that already have a `MessageSemanticRecord` are skipped (idempotent).

In [ ]:
from telegram_scraper.kg.services import (
    KGNodeProcessingService,
    KGNodeProjectionService,
    KGProcessingProgress,
)
from telegram_scraper.kg.event_hierarchy import KGEventHierarchyService

projection_service = KGNodeProjectionService(repository=repository, vector_store=vector_store)
hierarchy_service  = KGEventHierarchyService(repository)
node_service = KGNodeProcessingService(
    repository=repository,
    vector_store=vector_store,
    embedder=embedder,
    extractor=extractor,
    settings=kg_settings,
    projection_service=projection_service,
    hierarchy_service=hierarchy_service,
)
print("KGNodeProcessingService ready.")

In [ ]:
_t6 = time.monotonic()

def _progress(p: KGProcessingProgress):
    print(
        f"  [{p.messages_processed}/{p.messages_total}] "
        f"nodes={p.nodes_created} "
        f"assignments={p.assignments_created} "
        f"failures={p.failures} "
        f"rate={p.rate_per_sec:.1f}/s"
    )

process_result = node_service.process_messages(translated_messages, progress_callback=_progress)

_elapsed_extract = time.monotonic() - _t6
print(f"\nExtraction + resolution complete in {_elapsed_extract:.2f}s")
print(f"  assignments_created   : {process_result.assignments_created}")
print(f"  nodes_created         : {process_result.nodes_created}")
print(f"  cross_channel_matches : {process_result.cross_channel_matches}")
print(f"  relations_created     : {process_result.relations_created}")
print(f"  theme_stats_written   : {process_result.theme_stats_written}")

In [ ]:
# All nodes — sorted by article_count DESC.
all_nodes = repository.list_nodes(status=None)
node_rows = [
    {
        "node_id": n.node_id[:12] + "...",
        "kind": n.kind,
        "slug": n.slug,
        "display_name": n.display_name,
        "status": n.status,
        "articles": n.article_count,
    }
    for n in sorted(all_nodes, key=lambda n: -n.article_count)
]
nodes_df = pd.DataFrame(node_rows)
print(f"Total nodes: {len(nodes_df)}")
print("\nDistribution by kind:")
if not nodes_df.empty:
    print(nodes_df["kind"].value_counts().to_string())
nodes_df.head(25)

In [ ]:
# Assignments for the first translated message.
sample_msg = translated_messages[0]
sample_assignments = repository.list_message_node_assignments(
    message_keys=[(sample_msg.channel_id, sample_msg.message_id)]
)
print(
    f"Assignments for message {sample_msg.message_id} "
    f"(channel {sample_msg.channel_id}): {len(sample_assignments)}"
)
if sample_assignments:
    node_lookup = {n.node_id: n for n in all_nodes}
    assignment_rows = [
        {
            "node_id": a.node_id[:12] + "...",
            "kind": node_lookup[a.node_id].kind if a.node_id in node_lookup else "?",
            "display_name": node_lookup[a.node_id].display_name if a.node_id in node_lookup else "?",
            "confidence": round(a.confidence, 3),
            "is_primary_event": a.is_primary_event,
        }
        for a in sample_assignments
    ]
    pd.DataFrame(assignment_rows)

---
## Section 7 — Projection + Heat

1. `projection_service.refresh_all(days=31)` — rebuilds `node_relations` from message
   co-occurrence, writes `theme_daily_stats`, and refreshes `node_heat_view`.
2. Event hierarchy was already rebuilt inside `process_messages()`; this section
   re-runs projection explicitly for observability.
3. `classify_phase(snapshot, thresholds)` categorises each heat snapshot as
   `emerging` / `fading` / `sustained` / `flash_event` / `steady`.

In [ ]:
_t7 = time.monotonic()

projection_result = projection_service.refresh_all(days=31)
print(f"Projection done: relations={projection_result.relations_created}, theme_stats={projection_result.theme_stats_written}")

# Sample relations for the top node by article count.
if all_nodes:
    top_node = sorted(all_nodes, key=lambda n: -n.article_count)[0]
    relations = repository.list_node_relations(top_node.node_id)
    print(f"\nTop node: '{top_node.display_name}' ({top_node.kind}) — {len(relations)} relations")
    node_lookup_full = {n.node_id: n for n in all_nodes}
    rel_rows = [
        {
            "target": node_lookup_full[r.target_node_id].display_name if r.target_node_id in node_lookup_full else r.target_node_id[:12],
            "kind": node_lookup_full[r.target_node_id].kind if r.target_node_id in node_lookup_full else "?",
            "score": round(r.score, 3),
            "shared_messages": r.shared_message_count,
        }
        for r in sorted(relations, key=lambda r: -r.score)[:15]
    ]
    pd.DataFrame(rel_rows)

In [ ]:
from telegram_scraper.kg.heat_phase import (
    classify_phase,
    DEFAULT_THEME_HEAT_THRESHOLDS,
    DEFAULT_EVENT_HEAT_THRESHOLDS,
)

repository.refresh_node_heat_view()

theme_heat = repository.list_node_heat_rows(kind="theme")
event_heat = repository.list_node_heat_rows(kind="event")
print(f"Heat snapshots: {len(theme_heat)} themes, {len(event_heat)} events")

theme_thresholds = kg_settings.theme_heat_thresholds or DEFAULT_THEME_HEAT_THRESHOLDS
event_thresholds = kg_settings.event_heat_thresholds or DEFAULT_EVENT_HEAT_THRESHOLDS

heat_rows = []
for snap in theme_heat:
    heat_rows.append({
        "kind": snap.kind,
        "display_name": snap.display_name,
        "articles": snap.article_count,
        "heat_1d": round(snap.heat_1d, 4),
        "heat_7d": round(snap.heat_7d, 4),
        "heat_31d": round(snap.heat_31d, 4),
        "phase": classify_phase(snap, theme_thresholds),
    })
for snap in event_heat:
    heat_rows.append({
        "kind": snap.kind,
        "display_name": snap.display_name,
        "articles": snap.article_count,
        "heat_1d": round(snap.heat_1d, 4),
        "heat_7d": round(snap.heat_7d, 4),
        "heat_31d": round(snap.heat_31d, 4),
        "phase": classify_phase(snap, event_thresholds),
    })

heat_df = pd.DataFrame(heat_rows)
if not heat_df.empty:
    heat_df = heat_df.sort_values("articles", ascending=False).reset_index(drop=True)

_elapsed_projection = time.monotonic() - _t7
print(f"\nSection 7 done in {_elapsed_projection:.2f}s")
print("\nPhase distribution:")
if not heat_df.empty:
    print(heat_df["phase"].value_counts().to_string())
heat_df.head(20)

---
## Section 8 — Query + Grouped View

`KGQueryService` is the read-only query layer used by the Viz API.
`grouped_messages()` produces query-time message clusters bucketed by time window — this
replaces the old pre-computed `StoryUnit` abstraction and is produced on demand, not persisted.

In [ ]:
from telegram_scraper.kg.services import KGQueryService

query_service = KGQueryService(repository)

# Channels.
channels_in_db = query_service.channels()
print(f"Channels in DB: {len(channels_in_db)}")
pd.DataFrame([
    {"channel_id": c.channel_id, "title": c.channel_title, "messages": c.message_count}
    for c in channels_in_db
])

In [ ]:
# Top events, themes, people.
events = query_service.list_nodes(kind="event", limit=10)
themes = query_service.list_nodes(kind="theme", limit=10)
people = query_service.list_nodes(kind="person", limit=10)

print(f"Top events ({len(events)}):")
events_df = pd.DataFrame([
    {"slug": n.slug, "display_name": n.display_name, "articles": n.article_count}
    for n in events
])
print(events_df.to_string(index=False))

print(f"\nTop themes ({len(themes)}):")
themes_df = pd.DataFrame([
    {"slug": n.slug, "display_name": n.display_name, "articles": n.article_count}
    for n in themes
])
print(themes_df.to_string(index=False))

print(f"\nTop people ({len(people)}):")
people_df = pd.DataFrame([
    {"slug": n.slug, "display_name": n.display_name, "articles": n.article_count}
    for n in people
])
print(people_df.to_string(index=False))

In [ ]:
# Drill into a specific node.
NODE_KIND = "event"
NODE_SLUG = events[0].slug if events else ""

detail = None
if NODE_SLUG:
    detail = query_service.node_show(kind=NODE_KIND, slug=NODE_SLUG, message_limit=5)

if detail is None:
    print("No detail available — no events found or slug not matched.")
else:
    print(f"Node detail: '{detail.display_name}' (kind={detail.kind})")
    print(f"  summary          : {(detail.summary or '')[:120]}")
    print(f"  article_count    : {detail.article_count}")
    print(f"  related people   : {', '.join(n.display_name for n in detail.people[:5]) or '---'}")
    print(f"  related nations  : {', '.join(n.display_name for n in detail.nations[:5]) or '---'}")
    print(f"  related places   : {', '.join(n.display_name for n in detail.places[:5]) or '---'}")
    print(f"  related orgs     : {', '.join(n.display_name for n in detail.orgs[:5]) or '---'}")
    print(f"  related themes   : {', '.join(n.display_name for n in detail.themes[:5]) or '---'}")

    if detail.messages:
        msg_rows = [
            {
                "channel_title": nm.channel_title,
                "timestamp": nm.timestamp.strftime("%Y-%m-%d %H:%M"),
                "confidence": round(nm.confidence, 3),
                "text": (nm.english_text or nm.text or "")[:200],
            }
            for nm in detail.messages
        ]
        pd.DataFrame(msg_rows)

In [ ]:
# Grouped view — query-time message clusters by 1-day windows.
if detail is not None:
    groups = query_service.grouped_messages(node_id=detail.node_id, window="1d")
    print(f"{len(groups)} groups in 1-day windows for '{detail.display_name}'")
    for g in groups[:5]:
        print(f"  [{g.timestamp_start.date()}] {len(g.messages)} messages")
else:
    print("Skipped grouped_messages — no detail node available.")

---
## Section 9 — Summary

In [ ]:
from collections import Counter as _Counter

non_english_count = sum(
    1 for m in translated_messages
    if m.source_language and m.source_language not in ("en", "english")
)

kind_counts = _Counter(n.kind for n in all_nodes)
heat_count  = len(theme_heat) + len(event_heat)

print("=" * 60)
print("KG PIPELINE RUN SUMMARY")
print("=" * 60)
print(f"Channel          : {selected_chat.title} (id={selected_chat.chat_id})")
print(f"Messages ingested: {len(raw_messages)}")
print(f"  Non-English    : {non_english_count}")
print(f"Embeddings       : {len(embedding_records)}")
print(f"Nodes total      : {len(all_nodes)}")
for kind, count in sorted(kind_counts.items()):
    print(f"  {kind:<12}: {count}")
print(f"Assignments      : {process_result.assignments_created}")
print(f"Cross-channel    : {process_result.cross_channel_matches}")
print(f"Relations        : {projection_result.relations_created}")
print(f"Heat snapshots   : {heat_count}")
print()
print("Timing breakdown:")
print(f"  Setup          : {_elapsed_setup:.2f}s")
print(f"  Channel        : {_elapsed_channel:.2f}s")
print(f"  Ingestion      : {_elapsed_ingest:.2f}s")
print(f"  Translation    : {_elapsed_translate:.2f}s")
print(f"  Embedding      : {_elapsed_embed:.2f}s")
print(f"  Extraction     : {_elapsed_extract:.2f}s")
print(f"  Projection     : {_elapsed_projection:.2f}s")
_total = sum([
    _elapsed_setup, _elapsed_channel, _elapsed_ingest,
    _elapsed_translate, _elapsed_embed, _elapsed_extract, _elapsed_projection,
])
print(f"  TOTAL          : {_total:.2f}s")
print("=" * 60)

---
## Section 10 — Cleanup (Optional)

In [ ]:
# Disconnect the Telegram client when done.
await telegram_client.disconnect()
print("Telegram client disconnected.")

# ── Optional: clear all semantic state for the channel (keeps raw messages) ──
# Set RESET_CHANNEL = True to wipe nodes/assignments/semantics and re-run from scratch.
RESET_CHANNEL = False

if RESET_CHANNEL:
    result = repository.clear_semantic_state(channel_id=selected_chat.chat_id)
    print(f"Reset channel {selected_chat.chat_id}: {result}")
else:
    print("RESET_CHANNEL=False — semantic state preserved.")